In [8]:
import cv2
import tensorflow as tf
import numpy as np
from ultralytics import YOLO
from tensorflow.keras.applications.efficientnet import preprocess_input


In [9]:
hand_detection_path = r"C:\Users\PC\Desktop\test self\project\project asl with yolo\yolo hand model\best.pt"

asl_model_path = r"C:\Users\PC\Desktop\test self\project\project asl with yolo\best_asl_model.keras"

input_size = (224,224)

video_source = 0

crop_padding = 20

In [10]:
class_label = [chr(c) for c in range(ord("A"),ord("Z")+1)]+["del" , "nothing" , "space"]
print(class_label)

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [11]:
hand_detector = YOLO(hand_detection_path)
asl_classifier = tf.keras.models.load_model(asl_model_path)

c:\Users\PC\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:798: UserWarning: Skipping variable loading for optimizer 'adam', because it has 842 variables whereas the saved optimizer has 846 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [12]:
def preprocess_crop(crop):
    crop = cv2.cvtColor(crop , cv2.COLOR_BGR2RGB)
    crop = cv2.resize(crop , input_size)
    crop = crop.astype("float32")
    crop = preprocess_input(crop)
    crop = np.expand_dims(crop , axis=0)
    return crop


In [13]:
def classify_hand(crop):
    batch = preprocess_crop(crop)
    preds = asl_classifier.predict(batch)[0]
    class_idx = int(np.argmax(preds))
    confidence = float(preds[class_idx])
    label = class_label[class_idx] if class_idx < len(class_label) else "?"
    return label , confidence


In [14]:
def draw_prediction(frame , box , label , confidance):
    x1,y1,x2,y2 = box
    color = (0,200,0) if confidance >= 0.6 else (0,165,255)
    text = f"{label} ({confidance*100:.0f}%)" if confidance >= 0.6 else "?"
    cv2.rectangle(frame , (x1,y1) , (x2,y2) , color ,2)
    cv2.putText(frame , text , (x1 , max(y1-10,20)) , cv2.FONT_HERSHEY_SIMPLEX ,0.8 ,color , 2)

In [15]:
def main ():
    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print(f"couldnt open video source{video_source}")
        return

    frame_h , frame_w = None , None

    print("running , press 'q' to quit")

    while True :
        ok , frame = cap.read()
        if not ok :
            break

        if frame_h is None :
            frame_h , frame_w = frame.shape[:2]

        results = hand_detector.predict(frame , conf=0.6 , verbose=False)[0]

        for box in results.boxes:
            x1,y1,x2,y2 = map(int , box.xyxy[0].tolist())

            x1= max(0,x1 - crop_padding)
            y1 = max(0 ,y1 - crop_padding)
            x2 = min (frame_w , x2 + crop_padding)
            y2 = min (frame_h , y2 + crop_padding)

            crop = frame[y1:y2 , x1:x2]
            if crop.size==0 :
                continue

        label , confidance = classify_hand(crop)

        draw_prediction(frame , (x1,y1,x2,y2) , label , confidance)

        cv2.imshow("asl detection (yolo + efficientnetn4)" , frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


couldnt open video source0
